# Elasticsearch and OpenSearch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/elasticsearch.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Elasticsearch is a distributed search and analytics engine built on Apache Lucene. Documents are JSON, they live in indices split into shards and replicas, and nearly every query is answered from an inverted index rather than a table scan. OpenSearch is the Apache-2.0 fork of Elasticsearch 7.10.2, so the mental model and most of the API carry over; the differences are licensing, plugins, and feature velocity.

For a data engineer the framing is: Elasticsearch is a serving layer for text search, log analytics, faceted aggregation, and low-latency filtered lookups. It is not a system of record and not a warehouse, and treating it as either is the fastest route to an expensive, fragile cluster.

## What is Elasticsearch?

Elasticsearch stores JSON documents and answers two questions unusually fast: which documents contain these terms, and what are the counts and metrics grouped by these fields. It is built on Lucene segments, scales horizontally by splitting an index into shards, and tolerates node loss by keeping replicas on other nodes.

```bash
curl -s localhost:9200/_cluster/health?pretty
curl -s localhost:9200/_cat/indices?v
```

Name these in an interview: a cluster of nodes, indices made of primary and replica shards, a mapping of field types, analyzers that tokenize text, the inverted index for search, doc values for sorting and aggregation, and `_bulk` plus aliases for ingestion and zero-downtime changes. Since 7.11 the default distribution is Elastic License 2.0 / SSPL; OpenSearch forked the last Apache-2.0 release, 7.10.2.

Nodes have roles. Master-eligible nodes hold the cluster state, data nodes hold shards, ingest nodes run pipelines, and coordinating-only nodes fan out requests and merge results. The cluster state (index metadata, mappings, allocation decisions) is replicated across master-eligible nodes, so a stable master quorum is part of availability.

```bash
curl -s 'localhost:9200/_cat/nodes?v&h=ip,node.role,heap.percent,cpu,load_1m'
curl -s 'localhost:9200/_cluster/health?wait_for_status=yellow&timeout=30s&pretty'
```

The path of a search request: the coordinating node fans the query out to one copy of every shard of the target index, each shard scores its local top hits, and the coordinator merges them into the global top `size`. That is why search latency tracks the slowest shard rather than the average. A point read is different: `GET /index/_doc/id` routes to a single shard and can read the document before a refresh has made it searchable.

Common mistake: reading `_cluster/health` green as "every node is fast". Green only means all shards are allocated. Yellow means some replicas are unassigned, which is normal on a single node but a real risk on a multi-node cluster.

## How are indices shards and replicas organized?

An index is a logical namespace split into primary shards, each an independent Lucene index on one node. Replicas copy a primary onto another node, serve reads, and take over on failure. Primary shard count is fixed at creation; replicas can change at any time.

```bash
curl -X PUT localhost:9200/orders-v1 -H 'Content-Type: application/json' -d '{
  "settings": {"number_of_shards": 6, "number_of_replicas": 1, "refresh_interval": "5s"}
}'

curl -X PUT localhost:9200/orders-v1/_settings -H 'Content-Type: application/json' \
  -d '{"index": {"number_of_replicas": 2}}'
curl -s 'localhost:9200/_cat/shards/orders-v1?v'
```

Sizing rules: one shard per 10-50 GB of data, shard count a small multiple of the number of data nodes, and total shards per node in the low hundreds because each shard carries heap and file-handle overhead. Replicas cost disk and write bandwidth but buy availability and read throughput, and a primary and its replica never share a node.

A document lands on a shard by a hash of its routing value, which defaults to `_id`: `shard = hash(routing) % number_of_primary_shards`. That is why the primary shard count cannot change in place - a different count would route the same id to a different shard. A custom `?routing=` value co-locates related documents on one shard, which is how a tenant-scoped query avoids a fan-out.

Reads go to one copy of the shard and the coordinating node picks a primary or replica using adaptive replica selection, which prefers the least loaded copy. Writes always go to the primary, which replicates to the in-sync replicas and only then acknowledges, so `?wait_for_active_shards` and `?refresh=wait_for` trade latency for stronger guarantees.

```bash
curl -s 'localhost:9200/_cat/shards/orders-v1?v&h=index,shard,prirep,state,docs,store,node'
curl -s 'localhost:9200/orders-v1/_count?routing=t-88'
```

Common mistake: creating an index with one shard and zero replicas "to save space". Reads then cannot spread across nodes, and a single node restart or disk failure makes the index unavailable until it recovers. Keep at least one replica on any cluster with more than one node, and check `_cat/shards` for unassigned replicas after adding nodes.

## How do mappings work and what is the difference between keyword and text?

The mapping is the schema: it declares each field's type and how it is indexed. Indexing an identifier as `text` and then filtering or aggregating on it is the classic mistake. `text` is analyzed into tokens and searched with relevance; `keyword` is stored verbatim for exact filtering, sorting, and aggregation.

```json
{
  "mappings": {
    "properties": {
      "order_id": { "type": "keyword" },
      "status": { "type": "keyword" },
      "description": { "type": "text", "analyzer": "english" },
      "total": { "type": "scaled_float", "scaling_factor": 100 },
      "created_at": { "type": "date" },
      "shipping": { "properties": {
        "city": { "type": "keyword" }, "location": { "type": "geo_point" } } }
    }
  }
}
```

When a field must serve both purposes, use a multi-field: `"description": {"type": "text", "fields": {"raw": {"type": "keyword"}}}` indexes the value twice. Other types worth naming: `long`, `double`, `scaled_float` for money, `date`, `boolean`, `ip`, `geo_point`, `nested` for arrays of objects that must not cross-match, and `flattened` for arbitrary key/value blobs.

Every mapped field controls up to three separate structures. The inverted index powers search. `doc_values` is a columnar copy on disk used for sorting, aggregations, and `search_after`. `norms` stores the field length that BM25 uses to normalize scores. A `keyword` field has doc values and no analyzer; a `text` field has the inverted index and norms, and no doc values.

```json
{
  "mappings": {
    "properties": {
      "sku":       { "type": "keyword" },
      "trace_id":  { "type": "keyword", "index": false },
      "log_level": { "type": "keyword", "norms": false, "doc_values": false },
      "body":      { "type": "text", "norms": false },
      "notes":     { "type": "text", "doc_values": false }
    }
  }
}
```

Use `index: false` for a field you only return from `_source` and never query. Use `doc_values: false` for a field you never sort or aggregate on. Use `norms: false` on filter-only `text` fields. For a `keyword` field that holds long values, `ignore_above` stops indexing strings over a limit.

Common mistake: disabling `_source` to save disk. `_source` is what `update`, `reindex`, and highlighting read, so dropping it breaks those features. Use `"_source": {"excludes": ["large_field"]}` to trim it instead.

## What are analyzers and how do they affect search?

An analyzer runs at index time and again at query time: character filters, then a tokenizer, then token filters. The analyzers to know are `standard` (Unicode word boundaries, lowercasing), `keyword` (the whole value as one token), and `whitespace` for special cases. A mismatch between index and query analysis is a top cause of "the document exists but search returns nothing".

```bash
curl -s 'localhost:9200/_analyze?pretty' -H 'Content-Type: application/json' -d '{
  "analyzer": "standard", "text": "The 3 Quick Brown Foxes"
}'

curl -X PUT localhost:9200/docs -H 'Content-Type: application/json' -d '{
  "settings": {"analysis": {"analyzer": {"html_english": {
    "type": "custom", "tokenizer": "standard", "char_filter": ["html_strip"],
    "filter": ["lowercase", "stop", "porter_stem"] }}}},
  "mappings": {"properties": {"body": {"type": "text", "analyzer": "html_english"}}}
}'
```

The subtlety: `match` runs the analyzer on user input, but `term` does not, so `term` on a `text` field almost always fails to match. Use `match` for analyzed text and `term` only on `keyword`, numeric, or date values.

Walk the chain with `_analyze`. A character filter rewrites the raw text (`html_strip`), the tokenizer splits it (`standard`, `whitespace`, `keyword`, `letter`), and token filters normalize the tokens (`lowercase`, `stop`, `asciifolding`, `porter_stem`, `synonym`). Index and search analysis must agree, so a custom analyzer sets `analyzer` for indexing and `search_analyzer` for queries when they differ.

```bash
curl -s 'localhost:9200/_analyze?pretty' -H 'Content-Type: application/json' -d '{
  "tokenizer": "standard",
  "filter": ["lowercase", "stop", "porter_stem"],
  "text": "The Quick Brown Foxes were running"
}'
```

A term query does not run the analyzer, so it looks for the literal token. A match query does run it, and uses the field's `search_analyzer` (which defaults to the field's `analyzer`). Stemming at index time and not at query time is a classic mismatch: the index holds `run`, the term lookup asks for `running`, and the query returns nothing.

Common mistake: testing analysis with the global analyzer instead of the field's analyzer. Pass `"field": "body"` to `_analyze` to see the tokens the mapping actually produces.

```bash
curl -s 'localhost:9200/docs/_analyze?pretty' -H 'Content-Type: application/json' \
  -d '{"field": "body", "text": "The Quick Brown Foxes were running"}'
```

## What are the pitfalls of dynamic mapping?

With dynamic mapping on, the first document defines the types, and a mixed field can silently produce a mapping you cannot change. A `long` cannot later store a decimal, and a field first seen as a number cannot accept text without reindexing.

```bash
# This fails once total is mapped as a number
curl -X POST localhost:9200/orders/_doc -H 'Content-Type: application/json' \
  -d '{"total": "unknown"}'
# mapper_parsing_exception: failed to parse field [total] of type [float]
```

Mitigate by disabling or limiting dynamic mapping in the template (`"dynamic": "strict"` for known schemas, `"dynamic": false` to ignore unknown fields), pre-creating templates for every dataset, and routing unstructured payloads into a `flattened` field. Also remember that every string defaults to `text` plus a keyword sub-field, which doubles index storage; map ids and enums explicitly as `keyword`.

Dynamic mapping decides types from the first value it sees. A JSON string becomes `text` plus a `.keyword` sub-field, an integer becomes `long`, a decimal becomes `float`, an object becomes a nested `properties` tree, and a date-looking string can become `date` when it matches `dynamic_date_formats`. Setting `"dynamic": "strict"` rejects an unknown field at index time, which surfaces schema drift as an error instead of a mapping explosion.

```json
{
  "mappings": {
    "dynamic": "strict",
    "dynamic_templates": [
      { "strings_as_keywords": { "match_mapping_type": "string",
          "mapping": { "type": "keyword" } } }
    ],
    "properties": {
      "@timestamp": { "type": "date" },
      "message": { "type": "text" }
    }
  }
}
```

Set `"dynamic": false` to accept and store unknown fields without indexing them, or `"dynamic": "strict"` to reject them. Use `flattened` for an arbitrary key/value payload so one field absorbs everything instead of creating a field per key.

Common mistake: leaving dynamic mapping on for a payload with unbounded keys. Every new key adds a field, cluster state grows, and the index eventually hits `index.mapping.total_fields.limit` (default 1000). Check the damage with `GET /orders/_mapping` and the field count in `GET /orders/_stats`.

## How does the inverted index work?

For each analyzed field, Lucene writes a term dictionary mapping every token to the documents containing it, with frequencies and positions. A query looks up terms and intersects or unions posting lists, which is why full-text search is fast regardless of corpus size. Separately, `doc_values` store values column-wise for sorting and aggregation, and `_source` stores the original JSON.

```text
Document 1: "red running shoes"    Document 2: "blue running jacket"    Document 3: "red jacket"
```


Consequences: the inverted index only helps if the query uses indexed terms, which is why wildcards and leading-`*` regexes are slow (they scan terms) and why `text` fields cannot be sorted. Segment merges rewrite these structures, so indexing is I/O heavy and deletes are tombstones until a merge.

Worked example with the three documents above, analyzed by `standard`. The term dictionary holds `blue -> [2]`, `jacket -> [2,3]`, `red -> [1,3]`, `running -> [1,2]`, and `shoes -> [1]`. A `match` for `red jacket` becomes a boolean of two term lookups; the posting lists are intersected and the survivors are ranked by BM25. A `term` query for `Red` finds nothing, because the indexed token is lowercased to `red`.

Positions in the postings are what make `match_phrase` and proximity boosts possible, and the term frequencies are what BM25 scores. That is why `"index_options": "docs"` can save space on a field you only filter, at the cost of phrase queries.

Doc values and norms make the other half of the engine work. A `date_histogram` or `terms` aggregation reads `doc_values` in column order instead of touching the postings, and sorting does the same. That is why a `text` field cannot be aggregated directly: it has no doc values unless `fielddata: true` is set, which loads the whole field into the JVM heap.

## How do you index and update documents with the CRUD and bulk APIs?

Single-document calls are fine interactively; every pipeline should use `_bulk`, where each line is an action (`index`, `create`, `update`, `delete`) followed by the source, newline-delimited and without commas.

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

es = Elasticsearch("http://localhost:9200")


def actions(rows):
    for row in rows:
        yield {
            "_op_type": "update",
            "_index": "orders",
            "_id": row["order_id"],
            "doc": {"status": row["status"], "updated_at": row["updated_at"]},
            "doc_as_upsert": True,
        }


def load(rows) -> tuple[int, list]:
    return bulk(es, actions(rows), chunk_size=2000, request_timeout=120)

Indexing is near-real-time because a refresh makes new segments searchable, so a 30s `refresh_interval` and `refresh=false` during a backfill are much faster. A `200` from `_bulk` does not mean every item succeeded: inspect `errors` and retry `429`/`503` with backoff. Use `create` for exactly-once insertion and `update` with `doc_as_upsert` for idempotent upserts.

Indexing writes the document to an in-memory buffer and appends it to the translog for durability. A refresh builds a new Lucene segment from the buffer and opens a new searcher, which is what makes the document visible; the default `refresh_interval` is 1s. A flush persists the segments to disk and truncates the translog. The translog fsyncs per request by default (`index.translog.durability: request`); `async` with `sync_interval` trades a small window of loss for throughput.

```bash
curl -X PUT 'localhost:9200/orders/_doc/1001?refresh=wait_for' \
  -H 'Content-Type: application/json' -d '{"order_id":"1001","status":"paid"}'
curl -s 'localhost:9200/orders/_stats/refresh,flush,translog?pretty'
curl -s 'localhost:9200/orders/_cat/segments?v&h=index,shard,segment,docs.count,size'
```

Segment merging runs in the background. Lucene merges small segments into larger ones and reclaims deleted documents, which is why steady-state disk usage is lower than the raw write volume. Deletes and updates are tombstones until a merge. A `_bulk` call is newline-delimited JSON: an action line, then the source line, repeated, with no commas and a trailing newline.

```bash
curl -s -H 'Content-Type: application/x-ndjson' -X POST localhost:9200/_bulk --data-binary '
{"index":{"_index":"orders","_id":"1001"}}
{"order_id":"1001","status":"paid","total":42.50}
{"delete":{"_index":"orders","_id":"1000"}}
'
```

Common mistake: calling `_forcemerge` on a live index. It competes with indexing for I/O and produces very large segments that are expensive to update. Force-merge only read-only indices, ideally from ILM during the warm phase. A `200` from `_bulk` also does not mean every item succeeded: inspect `errors: true` and the per-item `status`, and retry `429` and `503` with backoff.

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

es = Elasticsearch("http://localhost:9200")


def actions(rows):
    for row in rows:
        yield {
            "_op_type": "update",
            "_index": "orders",
            "_id": row["order_id"],
            "doc": {"status": row["status"], "updated_at": row["updated_at"]},
            "doc_as_upsert": True,
        }


def load(rows):
    success, errors = bulk(
        es,
        actions(rows),
        chunk_size=2000,
        max_retries=5,
        initial_backoff=1,
        max_backoff=30,
        raise_on_error=False,
    )
    return success, errors


## How do you write search queries with match and term?

`match` analyzes the input and is the default for text; `term` matches one exact token and is for `keyword`, numbers, and booleans; `multi_match` searches several fields and combines their scores.

```json
{
  "query": {
    "bool": {
      "must": [
        { "multi_match": {
            "query": "wireless noise cancelling",
            "fields": ["title^3", "description", "brand^2"],
            "type": "best_fields"
        } }
      ],
      "filter": [
        { "term": { "status": "in_stock" } },
        { "range": { "price": { "gte": 50, "lte": 500 } } }
      ]
    }
  }
}
```

The `^3` boost makes title matches matter more, a first-order relevance lever before any ranking pipeline. `best_fields` scores the single best field, `most_fields` sums across fields, and `cross_fields` treats several fields as one, which is right for a first and last name searched together. Range queries support date math such as `"gte": "now-7d/d"`.

`match` with `"operator": "and"` requires every term, while the default `or` ranks any document that contains at least one. `match_phrase` uses the stored positions to keep order and adjacency. `prefix`, `wildcard`, and `regexp` walk the term dictionary and can expand to many terms, so a leading wildcard is a scan. `exists` tests whether the field has any indexed value.

```json
{
  "query": {
    "bool": {
      "must": [ { "match": { "title": { "query": "espresso machine", "operator": "and" } } } ],
      "filter": [
        { "term": { "brand.keyword": "Breville" } },
        { "range": { "price": { "gte": 100 } } },
        { "exists": { "field": "released_at" } }
      ]
    }
  }
}
```

You can also run the same analyzer endpoint against the query text to confirm what `match` will look up. Short fields and long fields score differently because BM25 normalizes by length, which is why a boost on `title` usually beats a boost on `description`.

Common mistake: `{"term": {"description": "Espresso"}}` on a `text` field. The indexed token is `espresso` and the query token is `Espresso`, so it misses. Use `match`, or query the exact `description.keyword` sub-field when you truly mean the whole value.

## How do bool queries combine clauses?

A `bool` query has four clause types: `must` (scores and filters), `filter` (filters, no score, cacheable), `should` (optional, contributes score, and with `minimum_should_match` becomes required), and `must_not` (excludes).

```json
{
  "query": {
    "bool": {
      "must": [ { "match": { "description": "espresso machine" } } ],
      "filter": [
        { "term": { "category": "kitchen" } },
        { "terms": { "warehouse": ["DFW", "PHX"] } },
        { "range": { "stock": { "gt": 0 } } }
      ],
      "should": [
        { "term": { "tags": "coffee" } },
        { "term": { "tags": "sale" } }
      ],
      "minimum_should_match": 1,
      "must_not": [ { "term": { "status": "discontinued" } } ]
    }
  }
}
```

The usual shape is `must` for relevance text, `filter` for yes/no attributes, and `should` for boosts. Because `filter` clauses skip scoring and are cacheable, moving numeric and term constraints out of `must` makes a query both faster and cache-friendly.

Scoring is the sum of the `must` and matching `should` clause scores; `filter` and `must_not` do not score. `minimum_should_match` defaults to 1 when the `bool` has no `must` or `filter`, and 0 otherwise, which surprises people who add one `should` and expect it to exclude documents.

```json
{
  "query": {
    "bool": {
      "must":   [ { "match": { "title": "espresso" } } ],
      "filter": [ { "term": { "status": "in_stock" } },
                  { "range": { "price": { "gte": 50 } } } ],
      "should": [ { "term": { "tags": "sale" } } ],
      "minimum_should_match": 0,
      "must_not": [ { "term": { "status": "discontinued" } } ]
    }
  }
}
```

Worked change: moving `{"term": {"status": "paid"}}` from `must` to `filter` matches exactly the same documents but removes its contribution to the score and lets the filter cache reuse the resulting bitset across queries. That is the single most common search-speed fix.

Common mistake: using a `should` clause to exclude documents. `should` only adds score unless `minimum_should_match` makes it required, and even then it only requires a match on one `should` clause, not on a specific one.

## What is the difference between queries and filters?

A query clause computes a BM25 `_score`, which is more expensive and depends on term statistics. A filter clause is a yes/no predicate: no score, and results cacheable in the filter cache. Elastic's own guidance is to filter anything that does not need to affect relevance.

```json
{
  "query": {
    "bool": {
      "must": [ { "match": { "title": "data engineering" } } ],
      "filter": [
        { "term": { "tenant_id": "t-88" } },
        { "term": { "doc_type": "article" } },
        { "range": { "published_at": { "gte": "2026-01-01" } } }
      ]
    }
  },
  "track_total_hits": false
}
```

Small choices compound: `track_total_hits: false` avoids counting every hit, `_source: false` or a `fields` list avoids shipping whole documents, and a selective `filter` lets the engine skip posting lists entirely.

The filter cache stores the doc-id bitset of a filter clause, keyed by the clause, so a repeated `term` or `range` filter becomes nearly free after the first run. That is why tenant, status, and time-range constraints belong in `filter` context. The related request cache stores the full response of an unchanged `size: 0` search, and its statistics are a quick health check.

```bash
curl -s 'localhost:9200/_nodes/stats/indices/query_cache,request_cache?pretty'
curl -s 'localhost:9200/orders/_search?pretty' -H 'Content-Type: application/json' -d '{
  "query": { "bool": { "filter": [
    { "term": { "status": "paid" } },
    { "range": { "created_at": { "gte": "now-7d/d" } } } ] } },
  "size": 0, "track_total_hits": false
}'
```

Common mistake: wrapping every constraint in `must` because it reads more simply. The matching set is identical, but the engine pays to score each clause and the filter cache never sees it. Profile with `"profile": true` and compare the time in the score phase against the build-scorer and next-doc phases to see the cost of scoring you did not need.

## How do aggregations work in Elasticsearch?

Aggregations compute metrics (`avg`, `sum`, `min`, `max`, `percentiles`, `cardinality`) and buckets (`terms`, `date_histogram`, `range`, `histogram`). Buckets nest, and a terms bucket can carry sub-aggregations to build a report in one request.

```json
{
  "size": 0,
  "query": { "range": { "created_at": { "gte": "now-7d/d" } } },
  "aggs": {
    "by_day": {
      "date_histogram": { "field": "created_at", "calendar_interval": "day",
        "time_zone": "America/Chicago" },
      "aggs": { "revenue": { "sum": { "field": "total" } },
        "p95": { "percentiles": { "field": "total", "percents": [50, 95, 99] } } }
    },
    "top_customers": {
      "terms": { "field": "customer_id", "size": 10, "order": { "revenue": "desc" } },
      "aggs": { "revenue": { "sum": { "field": "total" } } }
    },
    "unique_customers": { "cardinality": { "field": "customer_id" } }
  }
}
```

Caveats an interviewer probes: `terms` on a high-cardinality field is approximate - the top N per shard are merged, so `doc_count_error_upper_bound` can be non-zero; raise `shard_size` or use a composite aggregation to page all buckets. `cardinality` is approximate HyperLogLog++. `date_histogram` needs `time_zone` for local buckets, and `size: 0` is mandatory when only aggregations are wanted.

A `terms` aggregation is distributed and approximate. Each shard returns its local top `size * shard_size` buckets, the coordinator merges them, and the response reports `doc_count_error_upper_bound` and `sum_other_doc_count`. On a high-cardinality field across many shards the edge buckets can be wrong. A `composite` aggregation pages every bucket exactly using `after_key`, at the cost of a sorted walk and one request per page.

```json
{
  "size": 0,
  "aggs": {
    "by_tenant": {
      "composite": {
        "size": 1000,
        "sources": [
          { "tenant": { "terms": { "field": "tenant_id" } } },
          { "day": { "date_histogram": { "field": "created_at", "calendar_interval": "day" } } }
        ]
      },
      "aggs": { "revenue": { "sum": { "field": "total" } } }
    }
  }
}
```

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch("https://es.internal:9200")


def composite_buckets(index: str, page_size: int = 1000):
    after = None
    while True:
        body = {
            "size": 0,
            "aggs": {
                "by_tenant": {
                    "composite": {
                        "size": page_size,
                        "sources": [{"tenant": {"terms": {"field": "tenant_id"}}}],
                    }
                }
            },
        }
        if after:
            body["aggs"]["by_tenant"]["composite"]["after"] = after
        response = es.search(index=index, body=body)
        page = response["aggregations"]["by_tenant"]
        for bucket in page["buckets"]:
            yield bucket
        after = page.get("after_key")
        if not after:
            break


`cardinality` uses HyperLogLog++ and is approximate. `precision_threshold` (default 3000, max 40000) sets the point where counts become near-exact, and raising it costs memory. `date_histogram` needs a `time_zone` when buckets must follow a local calendar day.

Common mistake: aggregating on a `text` field. Terms, sort, and cardinality need doc values, a `text` field has none, and enabling `"fielddata": true` loads the field into the JVM heap. That is the classic cause of a search node running out of memory. Aggregate on the `keyword` field or a `.keyword` sub-field instead.

```bash
curl -s 'localhost:9200/_cat/fielddata?v&h=node,field,size'
curl -s 'localhost:9200/_nodes/stats/indices/fielddata?pretty'
```

## How does pagination work and what are the from and size limits?

Deep pagination with `from`/`size` is capped by `index.max_result_window` (default 10,000) because every shard fetches and sorts `from + size` hits. Beyond that, use `search_after` with a stable sort, pinned to a point-in-time (PIT) so the sort order cannot shift as documents are indexed.

```bash
curl -X POST 'localhost:9200/orders/_pit?keep_alive=2m'
# {"id":"48my..."}

curl -X POST localhost:9200/_search -H 'Content-Type: application/json' -d '{
  "size": 100,
  "pit": { "id": "48my...", "keep_alive": "2m" },
  "sort": [ { "created_at": "desc" }, { "_id": "asc" } ],
  "search_after": [ "2026-10-04T13:00:00.000Z", "1000" ]
}'

curl -X DELETE 'localhost:9200/_pit' -H 'Content-Type: application/json' -d '{"id":"48my..."}'
```

Use `scroll` only for bulk export; it holds segments open and is deprecated in favor of PIT plus `search_after`. For exports, writing a PIT result to Parquet is usually cheaper than paging it to a user, and `from`/`size` should stay shallow for interactive UIs.

`from`/`size` makes every shard build and sort `from + size` hits, so the page depth is capped by `index.max_result_window` (default 10000). `search_after` is stateless but needs a sort that is unique, so add `_id` or `_shard_doc` as a tiebreaker; otherwise a page can repeat or skip rows as the index changes. A point-in-time (PIT) freezes the segment view so `search_after` pages over a stable snapshot. `scroll` predates PIT, keeps a search context per shard, and is reasonable only for one-shot exports.

```bash
curl -X POST 'localhost:9200/orders/_pit?keep_alive=2m'
curl -X POST localhost:9200/_search -H 'Content-Type: application/json' -d '{
  "size": 100,
  "pit": { "id": "48my...", "keep_alive": "2m" },
  "sort": [ { "created_at": "desc" }, { "_shard_doc": "asc" } ],
  "search_after": [ "2026-10-04T13:00:00.000Z", 42 ]
}'
curl -X DELETE 'localhost:9200/_pit' -H 'Content-Type: application/json' -d '{"id":"48my..."}'
```

Common mistake: serving deep pages with `from`/`size` and raising `max_result_window` when the error appears. That multiplies heap and latency by the shard count and the page depth. Cap the UI at a few pages, give users filters instead, and paginate with `search_after` keyed on the last row's sort values. Close a PIT as soon as the export finishes; an open PIT keeps segments alive and blocks merges.

## What are index templates and how do you use them?

A template applies settings, mappings, and aliases to every index whose name matches a pattern. Composable and component templates let shared settings be defined once and overridden per dataset, which is how a logging platform creates correctly mapped daily indices without humans in the loop.

```bash
curl -X PUT localhost:9200/_index_template/logs-app -H 'Content-Type: application/json' -d '{
  "index_patterns": ["logs-app-*"],
  "priority": 200,
  "template": {
    "settings": {
      "number_of_shards": 3, "number_of_replicas": 1, "refresh_interval": "10s",
      "index.lifecycle.name": "logs-ilm",
      "index.default_pipeline": "logs-parse"
    },
    "mappings": {
      "dynamic": "strict",
      "properties": {
        "@timestamp": { "type": "date" }, "service": { "type": "keyword" },
        "level": { "type": "keyword" }, "message": { "type": "text" }
      }
    }
  }
}'
```

Templates are resolved by priority and pattern specificity, so overlapping templates are a common source of "why did my mapping come out wrong". `_simulate_index` shows the resolved settings before the first document arrives, and an ingest pipeline attached in the template parses every document automatically.

Resolution order: an index matches every template whose `index_patterns` match, the highest `priority` wins conflicts, and a more specific pattern breaks ties. Component templates are pulled in with `composed_of` and are not matched by name. Settings under `template.settings` are defaults, so a setting passed on the create call overrides them. Legacy `_template` and composable `_index_template` are separate APIs.

```bash
curl -s 'localhost:9200/_index_template/logs-app?pretty'
curl -s 'localhost:9200/_template?pretty'
curl -X POST 'localhost:9200/_index_template/_simulate_index/logs-app-2026.10.04?pretty'
```

Common mistake: defining `number_of_shards` in two overlapping templates and guessing which one won. `_simulate_index` prints the resolved settings and mappings for a concrete index name, which is the fastest way to debug a template. A template applies only at index creation, so an existing index keeps its old mapping until it is reindexed.

## How do aliases and reindex work?

An alias is a second name for one or more indices. Writing through an alias with `is_write_index` and searching through an alias that fans out over several indices is the standard pattern: applications never hard-code an index name, so a rebuild is invisible.

```bash
curl -X POST localhost:9200/_reindex?wait_for_completion=false -H 'Content-Type: application/json' -d '{
  "source": { "index": "orders-v1" },
  "dest": { "index": "orders-v2", "op_type": "create" }
}'

curl -X POST localhost:9200/_aliases -H 'Content-Type: application/json' -d '{
  "actions": [
    { "remove": { "index": "orders-v1", "alias": "orders" } },
    { "add": { "index": "orders-v2", "alias": "orders", "is_write_index": true } }
  ]
}'
```

`_reindex` copies documents and applies the destination mapping, so it is also the migration path after a mapping change - a field's type cannot be changed in place. It is not incremental by itself; for live data, pause writes or dual-write while the reindex catches up, then flip. `slices` parallelizes a large reindex.

Aliases can do more than rename. An alias with a `filter` presents a subset of an index, and an alias with `routing` pins queries to one shard. A write alias must name exactly one index with `is_write_index: true`. `_reindex` runs a search on the source and a bulk index on the destination, so `slices: auto` parallelizes it, `size` controls the batch, and `wait_for_completion=false` returns a task id to poll.

```bash
curl -X POST 'localhost:9200/_reindex?wait_for_completion=false&slices=auto' \
  -H 'Content-Type: application/json' -d '{
    "source": {"index": "orders-v1", "_source": ["order_id", "status", "total"]},
    "dest": {"index": "orders-v2"}
  }'
curl -s 'localhost:9200/_tasks?detailed=true&actions=*reindex*'
```

Common mistake: reindexing into a destination whose mapping is incompatible and discovering it halfway through. Read the `failures` array in the `_reindex` response, use `"conflicts": "proceed"` only when you understand the version conflicts it hides, and test the run against a small source `query` before launching the full job.

## How does index lifecycle management work?

ILM moves an index through hot, warm, cold, frozen, and delete phases based on age or size. Rollover creates a new backing index at a threshold, and each phase can change replicas, force-merge, move to cheaper nodes, or shrink.

```json
{
  "policy": {
    "phases": {
      "hot": { "actions": {
        "rollover": { "max_primary_shard_size": "50gb", "max_age": "1d" },
        "set_priority": { "priority": 100 } } },
      "warm": { "min_age": "2d", "actions": {
        "forcemerge": { "max_num_segments": 1 }, "set_priority": { "priority": 50 } } },
      "cold": { "min_age": "30d", "actions": {
        "readonly": {}, "searchable_snapshot": {}
      } },
      "delete": { "min_age": "365d", "actions": { "delete": {} } }
    }
  }
}
```

Attach it with `"index.lifecycle.name": "logs-ilm"` in the template and write through a rollover alias. Hot nodes need fast NVMe and CPU, warm nodes cheaper disk, and cold/frozen tiers keep data searchable from object storage. `searchable_snapshot` is the biggest cost lever for log retention because it removes the local copy.

A phase transition is driven by age or size. The ILM job polls indices, checks the phase `min_age`, and runs the phase's actions in a fixed order (rollover, then shrink or force-merge, then allocation). A rollover needs a rollover alias or data stream to point the write target at the new backing index. `_ilm/explain` shows the real age and the next action, and `?dry_run` shows whether an index would advance without changing it.

```bash
curl -X PUT localhost:9200/_ilm/policy/logs-ilm \
  -H 'Content-Type: application/json' -d @policy.json
curl -s 'localhost:9200/logs-app-000001/_ilm/explain?pretty'
curl -s 'localhost:9200/_ilm/status?pretty'
```

Common mistake: assuming `min_age: "30d"` on the delete phase means 30 days from index creation. For an index that rolls over, ages after the hot phase are measured from the rollover time, so the delete age is relative to the current backing index. Use `_ilm/explain` to confirm the age and the phase it is actually in.

```json
{
  "policy": {
    "phases": {
      "hot":  { "actions": { "rollover": { "max_primary_shard_size": "50gb", "max_age": "1d" } } },
      "warm": { "min_age": "2d",  "actions": { "forcemerge": { "max_num_segments": 1 }, "set_priority": { "priority": 50 } } },
      "cold": { "min_age": "30d", "actions": { "readonly": {}, "searchable_snapshot": {} } },
      "delete": { "min_age": "365d", "actions": { "delete": {} } }
    }
  }
}
```

## How do you ingest data with Logstash Beats and Kafka Connect?

There are four common paths: Beats agents ship logs and metrics; Logstash parses, enriches, and fans out; Kafka Connect sinks records from Kafka; and small services call the bulk API directly. For a data platform, Kafka plus a sink is usually most robust because Kafka absorbs spikes and replay handles index rebuilds.

```yaml
input:
  kafka:
    bootstrap_servers: "broker1:9092,broker2:9092"
    topics: ["logs.app"]
    codec: json
filter:
  date:
    match: ["ts", "ISO8601"]
    target: "@timestamp"
output:
  elasticsearch:
    hosts: ["https://es.internal:9200"]
    index: "logs-app-%{+YYYY.MM.dd}"
    ilm_enabled: true
    manage_template: true
```

```text
Beats --> Logstash --> Elasticsearch
Kafka --> Kafka Connect (Elasticsearch sink, bulk, retries) --> Elasticsearch
CDC (Debezium) --> Kafka --> sink --> Elasticsearch
```

Kafka Connect's sink batches, retries, and supports exactly-once with a dead-letter queue for rejected records. Use a business key as the document `_id` (for example `order_id`) so replays overwrite rather than duplicate, and always validate an ingest pipeline with `_ingest/pipeline/_simulate` before pointing production traffic at it.

An ingest pipeline is a chain of processors run before indexing: `grok` parses, `date` sets `@timestamp`, `remove` and `rename` tidy fields, `geoip` and `enrich` add context, and `script` does the rest. A pipeline can be the index default (`index.default_pipeline`) or the final one (`index.final_pipeline`, useful for a last timestamp). Test with `_simulate` and watch `_nodes/stats/ingest` for failures.

```bash
curl -X PUT localhost:9200/_ingest/pipeline/logs-parse -H 'Content-Type: application/json' -d '{
  "processors": [
    { "grok": { "field": "message", "patterns": ["%{TIMESTAMP_ISO8601:ts} %{LOGLEVEL:level} %{GREEDYDATA:msg}"] } },
    { "date": { "field": "ts", "formats": ["ISO8601"], "target_field": "@timestamp" } },
    { "remove": { "field": "ts" } }
  ]
}'
curl -X POST 'localhost:9200/_ingest/pipeline/logs-parse/_simulate?pretty' \
  -H 'Content-Type: application/json' \
  -d '{"docs": [{"_source": {"message": "2026-10-04T13:00:00Z INFO started"}}]}'
```

Common mistake: running a `script` processor on every document in a hot path, or enriching inside the search application when an `enrich` processor already exists. Enrichment that needs external data belongs upstream in Kafka or in an `enrich` processor backed by an enrich index, not in a per-document script.

## How does OpenSearch differ from Elasticsearch?

OpenSearch is the Apache-2.0 fork of Elasticsearch 7.10.2 plus OpenSearch Dashboards. Core concepts, mapping types, the DSL, and the bulk API are compatible enough that knowledge transfers; the differences are at the edges.

| Aspect | Elasticsearch | OpenSearch |
|---|---|---|
| License | Elastic License 2.0 / SSPL | Apache License 2.0 |
| Origin | Elastic / Lucene | AWS-led fork of 7.10.2 |
| Vector search | dense_vector, kNN, ELSER | kNN, neural search plugin |
| SQL | SQL, ESQL | SQL and PPL |
| Alerting | Kibana alerting | Alerting plugin, anomaly detection |
| Dashboards | Kibana | OpenSearch Dashboards |
| Managed | Elastic Cloud, AWS (limited) | AWS OpenSearch Service, self-hosted |

```text
GET _cluster/health                 # same in both
POST _plugins/_knn/{index}/_search  # OpenSearch plugin endpoint
```

OpenSearch wins when the license, an AWS-native managed service, or PPL/SQL querying matters; Elasticsearch wins on the latest relevance and vector features. Both keep the same operational shape - shards, mappings, templates, ILM (ISM in OpenSearch), and snapshots - so a migration is mostly reindexing and re-creating dashboards and alerts, not re-learning search.

Practical migration notes: the query DSL, mappings, bulk, and most aggregations are compatible, but plugin endpoints differ. OpenSearch uses `_plugins/_knn` for kNN, `_plugins/_security` for security, and ISM (`_plugins/_ism/policies`) instead of ILM. Snapshot restore across the two forks is not supported, so a migration is a reindex from a shared source such as the Parquet archive or the Kafka topic.

```bash
curl -s 'localhost:9200/_plugins/_ism/policies?pretty'   # OpenSearch
curl -s 'localhost:9200/_ilm/policy?pretty'              # Elasticsearch
```

Common mistake: assuming a client upgrade is transparent. The Elasticsearch 8 Python client rejects a server that reports as OpenSearch unless compatibility mode is set, and OpenSearch keeps its own client line. Pin the client to the server version and test the product check during the migration, not in production.

## How do you size an Elasticsearch cluster and plan shards?

Sizing starts from data volume, retention, and query shape. Estimate raw JSON size, multiply for `_source` plus indexes, add replicas and headroom, and divide into shards of 10-50 GB. Memory is dominated by the JVM heap (cap around 31 GB to keep compressed ordinary object pointers) and by the filesystem cache, which should hold the hot working set.

```text
1 TB/day raw JSON, 30-day retention
  stored/day          ~ 1 TB * 0.40 compression = 400 GB primary
  with one replica    ~ 800 GB/day
  30 days             ~ 24 TB per replica set (before cold/frozen offload)
  shards at 40 GB     ~ 10 primary shards/day, rolled over daily
```


Rules that survive production: make shards a few tens of GB, avoid tens of thousands of tiny daily shards, roll over at `max_primary_shard_size` rather than a fixed daily count when volume varies, and keep at least one replica. Over-sharding is far more common than under-sharding and costs heap, merge threads, and cluster state.

The heap is the first constraint. Set the JVM heap to at most half of RAM and at most about 31 GB so compressed ordinary object pointers stay on; the rest of RAM is filesystem cache for Lucene segments. A rough planning figure is 20 shards per GB of heap, but the real limit is the hot working set: if the recently searched segments do not fit in page cache, every search becomes a disk read. Count shards including replicas, and remember each shard carries segments, a translog, and cluster-state metadata.

```bash
curl -s 'localhost:9200/_cat/allocation?v&h=node,shards,disk.used_percent,disk.avail'
curl -s 'localhost:9200/_nodes/stats/jvm,fs,indices/segments?pretty'
```

Common mistake: sizing only by storage and ignoring the shard count. Ten thousand 1 GB shards fit on disk but not in heap, file handles, or merge threads. Use data streams with rollover at `max_primary_shard_size`, target 10-50 GB per primary shard, and move old data to a frozen tier or delete it instead of keeping thousands of small live shards.

## How do you profile and tune query performance?

Start with the slow log and the profile API, not guesswork. Search is slow for recurring reasons: a wildcard or script query, deep `from`/`size`, a high-cardinality `terms` aggregation, a field without `doc_values`, or a query that fans out to every shard.

```bash
curl -X PUT localhost:9200/orders/_settings -H 'Content-Type: application/json' \
  -d '{"index.search.slowlog.threshold.query.warn": "2s"}'

curl -X POST 'localhost:9200/orders/_search?pretty' -H 'Content-Type: application/json' \
  -d '{"profile": true, "query": {"bool": {"must": [{"match": {"description": "espresso"}}]}}}'

curl -s 'localhost:9200/_nodes/stats/indices/search,indices/query_cache?pretty'
```

Fixes by payoff: move non-scoring constraints into `filter`, replace wildcards with an analyzed field or `keyword` prefix, add `track_total_hits: false`, reduce `size` and use `search_after`, pre-aggregate with transforms or rollups, and raise `refresh_interval` during heavy ingestion. Force-merging read-only indices reduces segment count and improves latency, but is expensive and only safe on indices no longer written.

Diagnose with data, in order. `_nodes/stats/indices/search,query_cache,request_cache,fielddata` shows cache sizes and evictions. `_nodes/hot_threads` shows what the JVMs are actually doing. The slow log records the query body, and the profile API breaks one query into rewrite, collection, and aggregation phases. Rejections in `_cat/thread_pool` mean the search or write queue is full.

```bash
curl -s 'localhost:9200/_cat/thread_pool/search,write?v&h=node,name,active,queue,rejected'
curl -s 'localhost:9200/_nodes/hot_threads?threads=5'
curl -s 'localhost:9200/_nodes/stats/indices/request_cache,query_cache,fielddata?pretty'
```

Common mistake: tuning by raising thread pool queue sizes. A rejected search means requests arrive faster than the cluster can serve them, and a bigger queue only increases latency. Reduce per-query cost with filters, fewer shards, and `track_total_hits: false`; add capacity; or shed load at the edge.

## How do snapshots and backups work?

A snapshot is a point-in-time copy of indices and cluster state into a repository such as S3, GCS, Azure Blob, or a shared filesystem. Snapshots are incremental, so unchanged segments are referenced rather than recopied, which makes frequent snapshots cheap. It is the only sanctioned backup mechanism; copying data directories is not.

```bash
curl -X PUT localhost:9200/_snapshot/s3-backups -H 'Content-Type: application/json' \
  -d '{"type": "s3", "settings": {"bucket": "acme-es-backups", "base_path": "prod"}}'

curl -X PUT 'localhost:9200/_snapshot/s3-backups/snap-2026-10-04?wait_for_completion=true' \
  -H 'Content-Type: application/json' -d '{
    "indices": "orders,logs-app-*", "include_global_state": true, "ignore_unavailable": true
  }'

curl -X POST localhost:9200/_snapshot/s3-backups/snap-2026-10-04/_restore \
  -H 'Content-Type: application/json' \
  -d '{"indices": "orders", "rename_pattern": "orders", "rename_replacement": "orders_restored"}'
```

Add a `snapshot` action to ILM for long retention, and run a snapshot lifecycle policy (SLM) on a schedule with a retention count. Always test a restore into a differently named index, and keep the repository in a separate account or region so one credential mistake cannot remove both copies.

Why snapshots are cheap and safe: a snapshot walks the immutable segments of each shard, so a segment file already in the repository is only referenced, never recopied. Deleting one snapshot does not delete files still referenced by another. `_snapshot/_status` shows the in-flight shard counts, and `max_snapshot_bytes_per_sec` throttles the upload so backups do not starve queries. A restore always creates new indices and never overwrites an existing one.

```bash
curl -s 'localhost:9200/_snapshot/s3-backups/_status?pretty'
curl -X PUT localhost:9200/_slm/policy/nightly -H 'Content-Type: application/json' -d '{
  "schedule": "0 30 1 * * ?", "name": "<nightly-{now/d}>",
  "repository": "s3-backups",
  "config": {"indices": ["*"], "include_global_state": true},
  "retention": {"expire_after": "30d", "min_count": 7, "max_count": 60}
}'
```

Common mistake: treating a snapshot as searchable data. A snapshot must be restored into indices before it can be queried, which is why long retention uses `searchable_snapshot` in the cold or frozen phase, or a Parquet archive. Snapshots are for recovery and compliance, not for day-to-day search. Store the repository in a separate account or region so one credential mistake cannot remove both copies.

## How do you secure an Elasticsearch cluster?

Security has four layers: transport and HTTP TLS, authentication, authorization (roles plus document and field-level access), and audit logging. Exposing an unauthenticated cluster to a network is a breach waiting for a scanner.

```yaml
xpack.security.enabled: true
xpack.security.transport.ssl.enabled: true
xpack.security.transport.ssl.verification_mode: certificate
xpack.security.transport.ssl.keystore.path: certs/elastic-certificates.p12
xpack.security.http.ssl.enabled: true
xpack.security.http.ssl.keystore.path: certs/http.p12
```

```bash
curl -X POST localhost:9200/_security/role/logs_reader -H 'Content-Type: application/json' \
  -d '{"indices": [{"names": ["logs-app-*"], "privileges": ["read"]}]}'

curl -X POST localhost:9200/_security/role/tenant_t88 -H 'Content-Type: application/json' \
  -d '{"indices": [{"names": ["tickets-*"], "privileges": ["read"],
        "query": {"term": {"tenant_id": "t-88"}}}]}'
```

Beyond TLS and roles: API keys rather than shared passwords for services, field-level security and masked fields for PII, the audit log shipped off-cluster, encrypted snapshot repositories, and the cluster bound to a private network. When one index holds several tenants, document-level security plus a strict template is the isolation mechanism until tenants move to separate indices.

Roles combine cluster privileges, index privileges, and, with document-level security, a query. API keys scope a token to specific indices and privileges and can be invalidated without rotating a password. Field-level security and `masked_fields` control what a role sees inside `_source`, while an alias with a filter gives a coarser version of the same idea.

```bash
curl -X POST 'localhost:9200/_security/api_key?pretty' -H 'Content-Type: application/json' \
  -d '{"name": "log-shipper", "role_descriptors": {"shipper": {"index": [
        {"names": ["logs-app-*"], "privileges": ["create_doc", "auto_configure"]}]}}}'
curl -s 'localhost:9200/_security/_authenticate?pretty'
```

Common mistake: giving a service `all` on `*` because the exact privileges were unclear, then finding that the token can delete production indices. Follow least privilege: `create_doc` for a shipper, `read` on one pattern for a dashboard, and no cluster privilege unless it is genuinely needed. Bind Elasticsearch to a private network and never expose port 9200 to the internet.

## How do you design a log analytics architecture?

A log platform is a pipeline with a hot serving tier and a cold archive tier: producers write JSON with a timestamp and service identifier, shippers buffer and compress, a processing tier parses and enriches, and the index tier serves a bounded window while older data lives in object storage.

```text
apps --> Fluent Bit/Filebeat --> Kafka --> Logstash/Connect --> Elasticsearch
                                      |                              |
                                      |                              +-- ILM hot -> warm -> frozen -> delete
                                      +--> S3 archive (Parquet) --> Trino/Athena
```


Decisions that matter: normalize timestamp and service in the template so every dashboard works across indices; keep identifiers as `keyword` and free text as `text`; use `dynamic: strict` so a new field cannot blow up the mapping; set `refresh_interval` to 10-30s because dashboards do not need sub-second freshness; and sample or drop debug logs at the shipper. The Parquet archive is not optional - search clusters are expensive per byte, and Parquet answers compliance retention at a fraction of the cost.

A data stream is the modern shape for logs. It is a named write target backed by hidden, rolled-over indices, ILM handles rollover and retention, and the name always points at the writable backing index. The mapping is fixed by the template, so a new incompatible field fails at index time instead of silently changing the stream. A search over the stream fans out across the backing indices and their shards.

```bash
curl -X PUT localhost:9200/_data_stream/logs-app -H 'Content-Type: application/json' \
  -d '{"template": "logs-app-template"}'
curl -s 'localhost:9200/_data_stream/logs-app?pretty'
curl -s 'localhost:9200/_cat/indices/.ds-logs-app-*?v&h=index,docs.count,store.size'
```

Common mistake: letting a dashboard search the whole data stream with no time filter. Every refresh then fans out over every backing index and shard in the retention window. Constrain `@timestamp`, default the UI to a short window, and serve long-range charts from rollup or transform data rather than raw logs.

## How does search relevance work?

Relevance is the BM25 `_score`, which rewards term frequency, penalizes common terms, and normalizes by document length. The practical levers in order: get the analyzer right, use `multi_match` with field boosts, add `should` clauses for recency or popularity, use `function_score` for business signals, and only then consider learning-to-rank.

```json
{
  "query": {
    "function_score": {
      "query": { "multi_match": { "query": "espresso machine", "fields": ["title^3", "description"] } },
      "functions": [
        { "filter": { "term": { "in_stock": true } }, "weight": 2 },
        { "gauss": { "published_at": { "origin": "now", "scale": "30d", "decay": 0.5 } } }
      ],
      "score_mode": "sum",
      "boost_mode": "multiply"
    }
  }
}
```

Text relevance and analytics pull in opposite directions: the fields you analyze for search must also exist as `keyword` for aggregation. Measure relevance with a judgment set and nDCG or MRR rather than eyeballing, and use `explain: true` when a score is surprising. For zero-result queries, fall back to prefix or fuzzy matching (`fuzziness: "AUTO"`) and log the query so the gap is visible.

`explain` and judgments are how you tune relevance without guessing. `"explain": true` returns the BM25 term, idf, tf, and norm components for one hit. A judgment set pairs a query with known-good document ids, and nDCG or MRR scores a change before it ships. A `_score` is comparable only within a single query, never across queries.

```bash
curl -s 'localhost:9200/products/_search?pretty&explain=true' -H 'Content-Type: application/json' \
  -d '{"query": {"match": {"title": "espresso machine"}}, "size": 1}'
```

Common mistake: tuning relevance on a handful of queries you remember. A boost that fixes one query quietly breaks a hundred others. Keep the queries and expected ids in version control, rerun them after every analyzer or boost change, and track the zero-result rate as a first-order health metric.

## Elasticsearch vs ClickHouse vs Parquet and Trino?

All three answer analytical questions at different points on the latency/retention/operability curve. The choice follows the query shape: text search and faceted navigation go to Elasticsearch, high-concurrency columnar aggregation to ClickHouse, and cheap large-scale retention to Parquet queried by Trino.

| Dimension | Elasticsearch / OpenSearch | ClickHouse | Parquet + Trino |
|---|---|---|---|
| Storage | inverted index + doc values | columnar MergeTree parts | columnar immutable files |
| Best at | text search, facets, logs | sub-second aggregations | large scans, open data |
| Full-text | excellent, analyzers, relevance | basic token indexes | none natively |
| Updates | per document, near-real-time | async mutations | rewrite or delete files |
| Concurrency | high for search, agg-heavy costly | very high for aggregations | high via a cluster |
| Storage cost | highest | medium | lowest |
| Retention | ILM tiers, expensive per byte | local disks or S3 tiered | effectively unlimited |
| SQL | SQL/ESQL, limited joins | strong SQL, limited joins | full SQL and joins |
| Ops | JVM cluster, shard tuning | self-managed tuning | stateless query layer + metastore |

```text
search + logs -> Elasticsearch    aggregations -> ClickHouse    history -> Parquet + Trino
```

A defensible architecture uses all three: Kafka to Parquet for cheap immutable history, Trino or Spark for batch analytics over it, ClickHouse or a warehouse for interactive dashboards, and Elasticsearch for search, log exploration, and faceted filtering. Choosing one for everything is how teams end up with a search cluster doing warehousing.

Choosing is about the access pattern. If the dominant question is "which documents contain this text, and what are the facet counts", the inverted index wins. If it is "sum this metric over these dimensions across billions of rows", a columnar engine wins. If it is "scan this history cheaply and join it", files plus a query engine win. Each has a different cost curve per byte, per query, and per update.

Common mistake: making Elasticsearch the only store for a year of events because the search experience is good. Its storage cost per byte is the highest of the three, and wide aggregations compete with search for heap. Keep the expensive serving window short, archive to Parquet, and route analytical scans elsewhere.

## How do you choose shard count and size?

Shard size drives almost everything. A search touches every shard of the target index, so more shards means more fan-out, more merge work, and more heap and file handles. Target 10-50 GB per primary shard for time-series data, and keep total shards per node in the low hundreds. The primary count is fixed at index creation, so the practical answer is many small time-bounded indices rolled over by size, not one index with a guessed shard count.

```bash
curl -s 'localhost:9200/_cat/shards?v&h=index,shard,prirep,store,docs&s=store:desc' | head -20
curl -s 'localhost:9200/_cat/indices?v&h=index,pri,rep,docs.count,store.size&s=store.size:desc'
```

Worked example: 1 TB/day of raw logs at 0.4 compression is 400 GB/day of primary data. At 40 GB per shard that is 10 primary shards per day. With one replica each day holds 800 GB, and a 30-day window holds about 24 TB. Ten shards a day for 30 days is 600 primaries plus 600 replicas, which is already a lot of shards, so older days move to a frozen tier or are deleted sooner.

Common mistake: creating a daily index with a fixed shard count. Quiet days produce tiny shards and busy days produce hot ones. Roll over on `max_primary_shard_size` so shard count follows real volume, and check `_cat/shards` for shards under a few GB.

```python
import math

RAW_GB_PER_DAY = 1000
COMPRESSION = 0.40
TARGET_SHARD_GB = 40
RETENTION_DAYS = 30
REPLICAS = 1

primary_gb_per_day = RAW_GB_PER_DAY * COMPRESSION
shards_per_day = math.ceil(primary_gb_per_day / TARGET_SHARD_GB)
total_shards = shards_per_day * RETENTION_DAYS * (1 + REPLICAS)
print(f"primary GB/day: {primary_gb_per_day:.0f}")
print(f"shards/day: {shards_per_day}, total with replicas: {total_shards}")
assert primary_gb_per_day == 400.0
assert shards_per_day == 10
assert total_shards == 600
```

In [ ]:
import math

RAW_GB_PER_DAY = 1000
COMPRESSION = 0.40
TARGET_SHARD_GB = 40
REPLICAS = 1
HEAP_GB_PER_NODE = 31
MAX_SHARDS_PER_HEAP_GB = 20

primary_gb_per_day = RAW_GB_PER_DAY * COMPRESSION
shards_per_day = math.ceil(primary_gb_per_day / TARGET_SHARD_GB)
shard_budget_per_node = HEAP_GB_PER_NODE * MAX_SHARDS_PER_HEAP_GB
total_shards_at_30d = shards_per_day * 30 * (1 + REPLICAS)

print(f"primary GB/day: {primary_gb_per_day:.0f}")
print(f"shards/day at {TARGET_SHARD_GB} GB: {shards_per_day}")
print(f"shards after 30 days with {REPLICAS} replica(s): {total_shards_at_30d}")
print(f"shard budget per 31 GB heap node: {shard_budget_per_node}")

assert primary_gb_per_day == 400.0
assert shards_per_day == 10
assert total_shards_at_30d == 600
assert shard_budget_per_node == 620


## How do you tune mappings and analyzers?

Map identifiers, enums, and anything filtered or aggregated as `keyword`. Map prose as `text` with the right analyzer, and add a `.keyword` sub-field only when a sort or aggregation needs the full value. Change a field type by reindexing, because there is no in-place type change, so these decisions are worth making before the first document.

Disable what you do not use. `index: false` for a return-only field, `doc_values: false` for a field never sorted or aggregated, `norms: false` on filter-only text. Use `scaled_float` for money, `ignore_above` on high-cardinality keyword fields, and `doc_values` only where sorting or aggregating truly needs it.

```json
{
  "mappings": {
    "properties": {
      "order_id":  { "type": "keyword" },
      "status":    { "type": "keyword", "doc_values": true },
      "trace_id":  { "type": "keyword", "index": false },
      "notes":     { "type": "text", "norms": false },
      "user_agent": { "type": "keyword", "ignore_above": 512 },
      "total":     { "type": "scaled_float", "scaling_factor": 100 }
    }
  }
}
```

Common mistake: aggregating or sorting on a `text` field, hitting the fielddata error, and enabling `fielddata: true` to make it go away. That loads the field into the JVM heap and is the classic cause of an out-of-memory crash. Add a `keyword` sub-field instead. A multi-field also doubles indexing work, so add one only for a field you genuinely query both ways.

## How do you tune indexing and refresh?

The write path is buffer, translog, refresh, flush, merge. `refresh_interval` controls visibility, not durability: 30s or 60s instead of 1s cuts segment count and search overhead during a bulk load. Set `"refresh": false` on bulk requests during a backfill and refresh once at the end. Durability comes from the translog, so `index.translog.durability: async` risks only the sync window, not the whole load.

```bash
curl -X PUT localhost:9200/orders/_settings -H 'Content-Type: application/json' \
  -d '{"index": {"refresh_interval": "-1"}}'
curl -s -H 'Content-Type: application/x-ndjson' -X POST \
  'localhost:9200/_bulk?refresh=false' --data-binary @bulk.ndjson
curl -X POST 'localhost:9200/orders/_refresh'
```

Bulk sizing: 1,000 to 5,000 documents or about 5-15 MB per request is the usual sweet spot. Too small wastes round trips and creates many segments; too large risks memory pressure, long retries, and a single failure losing a big batch. Send bulk requests to several nodes rather than one. Replicas double the write work, so a backfill can temporarily set `number_of_replicas: 0` and restore it after. Merge tuning matters too: `index.merge.scheduler.max_thread_count: 1` reduces interference on spinning or network disks, and a larger `indices.memory.index_buffer_size` builds bigger segments before merge.

Common mistake: force-merging a live index or running a backfill with a 1s refresh and one replica. Both compete with search for CPU and I/O. Force-merge only read-only indices from ILM, raise `refresh_interval` during the load, and check indexing pressure with `_cat/thread_pool/write` and the merge counts in `_nodes/stats/indices/merges`.

## How do you tune search and aggregation performance?

Query shape comes first. Move non-scoring constraints into `filter`, avoid wildcards and regex on analyzed fields, avoid script queries in the hot path, and use `track_total_hits: false`. Reduce fan-out by constraining the time range and the set of indices, and reduce work per shard by lowering `size`. The profile API shows the split between scoring, collection, and aggregation.

```bash
curl -s 'localhost:9200/orders/_search?pretty' -H 'Content-Type: application/json' \
  -d '{"profile": true, "query": {"bool": {"filter": [{"term": {"status": "paid"}}]}}}'
curl -s 'localhost:9200/_nodes/hot_threads?threads=5'
```

For aggregations, prefer `keyword` fields with doc values, keep the terms `size` small, use `composite` when you need every bucket, and set `precision_threshold` on `cardinality` deliberately. Never paginate deeply with `from`/`size`; use PIT plus `search_after`. The request cache helps repeated identical `size: 0` searches and the query cache helps repeated filters, so watch their eviction counts: evictions mean the working set does not fit.

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch("https://es.internal:9200")


def page_with_pit(index: str, sort_field: str, page_size: int = 500):
    pit = es.open_point_in_time(index=index, keep_alive="2m")
    after = None
    try:
        while True:
            body = {
                "size": page_size,
                "pit": {"id": pit["id"], "keep_alive": "2m"},
                "sort": [{sort_field: "asc"}, {"_shard_doc": "asc"}],
            }
            if after:
                body["search_after"] = after
            response = es.search(body=body)
            hits = response["hits"]["hits"]
            if not hits:
                break
            yield from hits
            after = hits[-1]["sort"]
    finally:
        es.close_point_in_time(body={"id": pit["id"]})


Common mistake: tuning aggregation speed by raising the terms `size` or the `precision_threshold` and expecting exact answers. Both increase coordinator work and memory. Measure first with `_nodes/stats/indices/fielddata` and the profile API, then decide whether the extra precision is worth the cost.

In [ ]:
import math

SHARDS = 30
TERMS_SIZE = 100
SHARD_SIZE = int(TERMS_SIZE * 1.5) + 10
BYTES_PER_BUCKET = 96
DOCS = 2_000_000_000

coordinator_buckets = SHARDS * SHARD_SIZE
coordinator_bytes = coordinator_buckets * BYTES_PER_BUCKET
docs_per_shard = DOCS / SHARDS

print(f"candidate buckets per shard: {SHARD_SIZE}")
print(f"buckets merged at the coordinator: {coordinator_buckets:,}")
print(f"coordinator bucket memory: {coordinator_bytes / 1024:.1f} KiB")
print(f"documents per shard: {docs_per_shard:,.0f}")

assert SHARD_SIZE == 160
assert coordinator_buckets == 4800
assert coordinator_bytes == 460_800
assert round(docs_per_shard, 2) == 66666666.67


## Real world use cases for Elasticsearch

### 1. Log analytics and observability

Business context: 4,000 services emit 1.5 TB of logs/day; engineers search the last 7 days in under a second and compliance needs 12 months. Design: Filebeat to Kafka, Connect for parsing into a strict template, daily rollover indices with ILM, and a Parquet archive in S3.

```json
{
  "index_patterns": ["logs-app-*"],
  "template": {
    "settings": { "refresh_interval": "10s", "index.lifecycle.name": "logs-ilm" },
    "mappings": {
      "dynamic": "strict",
      "properties": {
        "@timestamp": { "type": "date" }, "service": { "type": "keyword" },
        "level": { "type": "keyword" }, "message": { "type": "text" }
      }
    }
  }
}
```

Why Elasticsearch: analyzed text search plus `terms` and `date_histogram` answer both "find this error" and "how many per service per hour", and ILM moves older data to cheaper tiers.

### 2. E-commerce search with faceted navigation

Business context: 8M SKUs, 12,000 searches/s, filters on brand, price, and rating, with continuous catalog updates. Design: `text` for title and description, `keyword` for brand, `scaled_float` for price, Kafka Connect with the SKU as `_id`, and an alias for reindex.

```json
{
  "size": 20,
  "query": {
    "bool": {
      "must": [ { "multi_match": { "query": "noise cancelling headphones",
        "fields": ["title^3", "brand^2", "description"] } } ],
      "filter": [ { "term": { "in_stock": true } },
        { "range": { "price": { "gte": 50, "lte": 400 } } } ]
    }
  },
  "aggs": {
    "brands": { "terms": { "field": "brand", "size": 20 } },
    "avg_rating": { "avg": { "field": "rating" } }
  }
}
```

Why Elasticsearch: one request returns ranked hits and the filter counts the UI needs, and boosting plus `function_score` lets merchandising influence order.

### 3. Security information and event management

Business context: a security team correlates events from 60 sources and needs continuous detection plus forensic search over 13 months. Design: normalized events, a write alias with rollover, ILM with a frozen tier, scheduled detection searches, and immutable snapshots.

```text
detection rule, every 5 minutes
  search:   event.category:authentication AND event.outcome:failure
  group by: source.ip
  trigger:  count > 20 within 5 minutes
```

Why Elasticsearch: high-cardinality keyword filtering plus aggregations make correlation practical, and each hit keeps its `_source` for investigation.

### 4. Product catalog sync from a lakehouse

Business context: the warehouse (Iceberg on S3, queried by Trino) is the system of record; the search index must reflect changes within minutes and survive a rebuild without downtime. Design: a curated Parquet export, a bulk-index job into a versioned index, and an alias flip.

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import streaming_bulk
import pyarrow.dataset as ds

es = Elasticsearch("https://es.internal:9200")


def documents(path: str, batch_size: int = 5000):
    dataset = ds.dataset(path, format="parquet")
    for batch in dataset.to_batches(batch_size=batch_size):
        for row in batch.to_pylist():
            yield {"_index": "products-v3", "_id": row["sku"], "_source": row}


def rebuild(path: str) -> tuple[int, list]:
    return streaming_bulk(es, documents(path), chunk_size=2000, max_retries=5)

Why Elasticsearch: bulk indexing with business-key ids is idempotent, the alias makes the swap invisible, and the lakehouse stays the single source of truth.

### 5. Multi-tenant customer support search

Business context: a SaaS helpdesk holds 900M tickets across 2,000 tenants, every query is tenant-scoped, and one tenant is 30% of the data. Design: a shared index with `tenant_id` as `keyword`, document-level security, routing by `tenant_id`, and per-contract ILM retention.

```json
{
  "query": {
    "bool": {
      "must": [ { "multi_match": { "query": "refund not received",
        "fields": ["subject^2", "body"] } } ],
      "filter": [
        { "term": { "tenant_id": "t-88" } },
        { "range": { "created_at": { "gte": "now-90d/d" } } }
      ]
    }
  },
  "highlight": { "fields": { "body": {} } }
}
```

```bash
curl -X POST 'localhost:9200/tickets/_doc/9001?routing=t-88' -H 'Content-Type: application/json' \
  -d '{"tenant_id":"t-88","subject":"Refund missing","body":"I have not received my refund."}'
```

Why Elasticsearch: document-level security plus routing gives tenant isolation and locality, highlight returns the snippet the UI needs, and per-tenant retention is an ILM concern.

## Elasticsearch exercises

These exercises use only the Python standard library and model shard, storage, mapping, and aggregation arithmetic. Each one names the task, gives a hint, states the expected result, then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are estimates, not promises. Real results depend on compression, cardinality, mapping, and cluster shape; the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Choose shard count from index size and growth

**Task.** An index holds 500 GB of primary data today and grows 8% per month. Target 40 GB per shard and plan for a 12-month horizon. Choose the number of primary shards.

**Hint.** Project the size with `current * (1 + growth) ** months`, then take `ceil(projected / target)`.

**Expected result.** About 1,259.1 GB projected, which rounds up to 32 primary shards.

In [ ]:
import math

CURRENT_GB = 500
MONTHLY_GROWTH = 0.08
HORIZON_MONTHS = 12
TARGET_SHARD_GB = 40

projected_gb = CURRENT_GB * (1 + MONTHLY_GROWTH) ** HORIZON_MONTHS
shards = math.ceil(projected_gb / TARGET_SHARD_GB)

print(f"projected primary data: {projected_gb:,.1f} GB")
print(f"primary shards at {TARGET_SHARD_GB} GB: {shards}")

assert round(projected_gb, 1) == 1259.1
assert shards == 32


### Try it 2. Compute node and storage needs from replicas and overhead

**Task.** A cluster holds 12 TB of primary data with one replica and keeps 30% of each disk free. Each node provides 8 TB of usable storage. Compute the logical size, the raw storage needed, and the node count.

**Hint.** `logical = primary * (1 + replicas)`; `raw = logical / (1 - free_fraction)`; `nodes = ceil(raw / usable_per_node)`.

**Expected result.** 24 TB logical, about 34.29 TB raw, and 5 nodes.

In [ ]:
import math

PRIMARY_TB = 12
REPLICAS = 1
FREE_SPACE = 0.30
USABLE_TB_PER_NODE = 8

logical_tb = PRIMARY_TB * (1 + REPLICAS)
raw_tb = logical_tb / (1 - FREE_SPACE)
nodes = math.ceil(raw_tb / USABLE_TB_PER_NODE)

print(f"logical with {REPLICAS} replica(s): {logical_tb:.0f} TB")
print(f"raw storage at {FREE_SPACE:.0%} free: {raw_tb:.2f} TB")
print(f"data nodes needed: {nodes}")

assert logical_tb == 24
assert round(raw_tb, 2) == 34.29
assert nodes == 5


### Try it 3. Decide keyword versus text from query patterns

**Task.** Choose a field type for four fields: `order_id` (filtered and sorted), `title` (full-text searched only), `status` (filtered only), and `description` (searched for text and aggregated as a whole value).

**Hint.** Full text needs `text`; exact filter or sort needs `keyword`; both at once needs a `text` field with a `keyword` sub-field.

**Expected result.** `keyword`, `text`, `keyword`, `text with keyword sub-field`.

In [ ]:
def choose_type(full_text, exact_use):
    if full_text and exact_use:
        return "text with keyword sub-field"
    if full_text:
        return "text"
    return "keyword"

fields = [
    ("order_id", False, True),
    ("title", True, False),
    ("status", False, True),
    ("description", True, True),
]
result = {name: choose_type(full_text, exact_use) for name, full_text, exact_use in fields}
for name, mapping in result.items():
    print(f"{name}: {mapping}")

assert result["order_id"] == "keyword"
assert result["title"] == "text"
assert result["status"] == "keyword"
assert result["description"] == "text with keyword sub-field"


### Try it 4. Detect oversharding from shard count and cluster state

**Task.** A 5-node cluster holds 96,000 GB across 120,000 shards. Flag the cluster as oversharded if the average shard is under 10 GB or any node holds more than 1,000 shards.

**Hint.** `average_gb = total_gb / shards`; `per_node = shards / nodes`.

**Expected result.** 0.8 GB per shard and 24,000 shards per node, so the cluster is oversharded.

In [ ]:
DATA_NODES = 5
SHARDS = 120_000
TOTAL_GB = 96_000
MIN_SHARD_GB = 10
MAX_SHARDS_PER_NODE = 1_000

average_gb = TOTAL_GB / SHARDS
per_node = SHARDS / DATA_NODES
oversharded = average_gb < MIN_SHARD_GB or per_node > MAX_SHARDS_PER_NODE

print(f"average shard: {average_gb:.2f} GB")
print(f"shards per node: {per_node:,.0f}")
print("oversharded:", oversharded)
print("fix: consolidate into fewer, larger indices or delete cold shards")

assert average_gb == 0.8
assert per_node == 24_000
assert oversharded


### Try it 5. Size a bulk request from document size

**Task.** Documents average 2 KB, and the target request body is 5 MB. Compute the documents per request and the number of requests needed to load 50 million documents.

**Hint.** `docs_per_request = floor(target_bytes / doc_bytes)`; `requests = ceil(total / docs_per_request)`.

**Expected result.** 2,560 documents per request and 19,532 requests.

In [ ]:
import math

DOC_BYTES = 2048
TARGET_MB = 5
TOTAL_DOCS = 50_000_000

docs_per_request = (TARGET_MB * 1024 * 1024) // DOC_BYTES
requests = math.ceil(TOTAL_DOCS / docs_per_request)

print(f"documents per bulk request: {docs_per_request:,}")
print(f"bulk requests for {TOTAL_DOCS:,} docs: {requests:,}")

assert docs_per_request == 2560
assert requests == 19532


### Try it 6. Estimate aggregation memory from cardinality

**Task.** A `cardinality` aggregation uses `precision_threshold = 40000` and the query hits 10 shards. HyperLogLog++ uses roughly `precision_threshold * 8` bytes per shard. Compute the per-shard and total memory, and compare with the default threshold of 3000.

**Hint.** `per_shard = threshold * 8`; `total = per_shard * shards`.

**Expected result.** 320,000 bytes (312.5 KiB) per shard, 3,200,000 bytes total, versus 24,000 bytes per shard at the default.

In [ ]:
PRECISION_THRESHOLD = 40_000
DEFAULT_THRESHOLD = 3_000
SHARDS = 10
BYTES_PER_PRECISION = 8

per_shard_bytes = PRECISION_THRESHOLD * BYTES_PER_PRECISION
total_bytes = per_shard_bytes * SHARDS
default_per_shard_bytes = DEFAULT_THRESHOLD * BYTES_PER_PRECISION

print(f"per shard: {per_shard_bytes:,} bytes ({per_shard_bytes / 1024:.1f} KiB)")
print(f"total across {SHARDS} shards: {total_bytes:,} bytes")
print(f"default per shard: {default_per_shard_bytes:,} bytes")

assert per_shard_bytes == 320_000
assert total_bytes == 3_200_000
assert default_per_shard_bytes == 24_000


### Try it 7. Plan an ILM rollover from daily index size

**Task.** Logs arrive at 600 GB/day of primary data. Target 50 GB per shard, keep 90 days, and use one replica. Compute shards per day, total shards including replicas, and the on-disk footprint.

**Hint.** `shards_per_day = ceil(daily_gb / target_gb)`; multiply by retention and `(1 + replicas)` for the shard total.

**Expected result.** 12 shards per day, 2,160 shards total, 105.5 TB on disk.

In [ ]:
PRIMARY_GB_PER_DAY = 600
TARGET_SHARD_GB = 50
RETENTION_DAYS = 90
REPLICAS = 1

shards_per_day = -(-PRIMARY_GB_PER_DAY // TARGET_SHARD_GB)
total_shards = shards_per_day * RETENTION_DAYS * (1 + REPLICAS)
footprint_tb = PRIMARY_GB_PER_DAY * RETENTION_DAYS * (1 + REPLICAS) / 1024

print(f"shards per day: {shards_per_day}")
print(f"total shards with replicas: {total_shards:,}")
print(f"on-disk footprint: {footprint_tb:.1f} TB")
print("roll over on max_primary_shard_size so quiet days make fewer shards")

assert shards_per_day == 12
assert total_shards == 2160
assert round(footprint_tb, 1) == 105.5


### Try it 8. Compute deep pagination cost per shard

**Task.** A search pages `from = 9000, size = 100` across 20 shards. Compute the hits each shard must sort, the total fetched, and whether the page is inside `max_result_window = 10000`. Then compute the same for `search_after` with `size = 100`.

**Hint.** `per_shard = from + size`; `total = shards * per_shard`; a page is allowed while `from + size <= max_result_window`.

**Expected result.** 9,100 hits per shard, 182,000 total, inside the window, and 2,000 hits with `search_after`.

In [ ]:
SHARDS = 20
FROM = 9000
SIZE = 100
MAX_RESULT_WINDOW = 10000

per_shard = FROM + SIZE
total_fetched = SHARDS * per_shard
within_window = per_shard <= MAX_RESULT_WINDOW
search_after_fetched = SHARDS * SIZE

print(f"hits sorted per shard: {per_shard:,}")
print(f"hits fetched across {SHARDS} shards: {total_fetched:,}")
print("within max_result_window:", within_window)
print(f"hits with search_after: {search_after_fetched:,}")

assert per_shard == 9100
assert total_fetched == 182_000
assert within_window
assert search_after_fetched == 2000
